# 안전한 최적화 (vLLM 호환 보장)

## 이전 에러 원인
```
RuntimeError: Engine core initialization failed
```

**원인**: lm_head 양자화 시 `tie_word_embeddings`가 `false`로 변경되어 vLLM 호환성 깨짐

## 해결책
- `ignore=["embed_tokens", "lm_head"]` 필수 유지
- 캘리브레이션 품질만 향상 (안전한 개선)

---

In [1]:
import os
import torch
import shutil

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.9.1
CUDA: False


In [2]:
# ============================================================================
# 안전한 설정 (vLLM 호환 보장)
# ============================================================================

MODEL_ID = "./open/base_model"
OUT_DIR = "./model"
DATASET_ID = "LGAI-EXAONE/MANTA-1M"

# 캘리브레이션 강화 (안전한 개선)
NUM_SAMPLES = 512       # 256 → 512
MAX_SEQ_LEN = 1024      # 512 → 1024

# 양자화 설정 (베이스라인 + 최적화)
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE = ["embed_tokens", "lm_head"]  # 필수! vLLM 호환

# GPTQ 최적화 파라미터
BLOCK_SIZE = 128        # Marlin 호환
DAMPENING = 0.001       # Hessian 안정화
ACTORDER = "weight"     # 정확도 향상

print("=" * 60)
print("안전한 최적화 설정")
print("=" * 60)
print(f"캘리브레이션: {NUM_SAMPLES}샘플, {MAX_SEQ_LEN}길이")
print(f"ignore: {IGNORE} (필수!)")
print(f"actorder: {ACTORDER}")
print("=" * 60)

안전한 최적화 설정
캘리브레이션: 512샘플, 1024길이
ignore: ['embed_tokens', 'lm_head'] (필수!)
actorder: weight


In [3]:
print("[1/5] 모델 로드...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

print(f"  파라미터: {model.num_parameters():,}")

`torch_dtype` is deprecated! Use `dtype` instead!


[1/5] 모델 로드...
  파라미터: 1,279,391,488


In [4]:
print(f"[2/5] 데이터셋 로드 ({NUM_SAMPLES}개)...")

ds = load_dataset(DATASET_ID, split=f"train[:{NUM_SAMPLES}]")

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)
print(f"  완료: {len(ds)}개")

[2/5] 데이터셋 로드 (512개)...
  완료: 512개


In [5]:
print("[3/5] GPTQ 양자화...")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,  # embed_tokens, lm_head 제외!
        block_size=BLOCK_SIZE,
        dampening_frac=DAMPENING,
        actorder=ACTORDER,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQ_LEN,
    num_calibration_samples=NUM_SAMPLES,
)

print("  완료!")

[3/5] GPTQ 양자화...


Tokenizing:   0%|          | 0/512 [00:00<?, ? examples/s]

2026-02-11T23:15:59.023084+0900 | reset | INFO - Compression lifecycle reset
2026-02-11T23:15:59.024174+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-11T23:15:59.044066+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-11T23:15:59.044468+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-11T23:15:59.049828+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


W0211 23:15:59.072000 10291 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/31): Calibrating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [07:39<00:00,  1.11it/s]

2026-02-11T23:23:38.899517+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 512 samples


...

(30/31): Calibrating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [05:31<00:00,  1.54it/s]

2026-02-12T04:18:17.250016+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 512 samples


2026-02-12T04:18:17.540298+0900 | compress | METRIC - time 0.29s
2026-02-12T04:18:17.540680+0900 | compress | METRIC - error 5645.26
2026-02-12T04:18:17.541492+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-12T04:18:17.541714+0900 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-12T04:18:17.543506+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 512 samples
2026-02-12T04:18:17.724576+0900 | compress | METRIC - time 0.18s
2026-02-12T04:18:17.724917+0900 | compress | METRIC - error 1600.08
2026-02-12T04:18:17.725714+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-12T04:18:17.725935+0900 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-12T04:18:17.726704+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 512 samples
2026-02-12T04:18:17.907993+0900 | compress | METRIC - time 0.18s
2026-02-12T04:1

(31/31): Propagating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [00:00<00:00, 1794.45it/s]


2026-02-12T04:22:46.174357+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-12T04:22:46.180081+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
  완료!


In [6]:
print("[4/5] 모델 저장...")

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

# 크기 확인
total = sum(os.path.getsize(os.path.join(OUT_DIR, f)) for f in os.listdir(OUT_DIR))
print(f"  크기: {total/1e9:.2f} GB")

[4/5] 모델 저장...
2026-02-12T04:22:46.446853+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:01, 181.92it/s]


  크기: 0.98 GB


In [7]:
# config.json 검증
import json

with open(f"{OUT_DIR}/config.json") as f:
    cfg = json.load(f)

print("\n[검증] config.json")
print(f"  tie_word_embeddings: {cfg.get('tie_word_embeddings')}")
print(f"  ignore: {cfg.get('quantization_config', {}).get('ignore')}")

# 검증
if cfg.get('tie_word_embeddings') == True:
    print("\n  tie_word_embeddings 유지됨")
else:
    print("\n  경고: tie_word_embeddings 변경됨!")


[검증] config.json
  tie_word_embeddings: True
  ignore: ['lm_head']

  tie_word_embeddings 유지됨


In [8]:
print("[5/5] 제출 파일 생성...")

zip_name = "submit_safe"
if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")

shutil.make_archive(zip_name, "zip", ".", OUT_DIR)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9

print("\n" + "=" * 60)
print("제출 준비 완료!")
print("=" * 60)
print(f"파일: {zip_name}.zip")
print(f"크기: {zip_size:.2f} GB")
print("=" * 60)

[5/5] 제출 파일 생성...

제출 준비 완료!
파일: submit_safe.zip
크기: 0.81 GB


---

## 개선 포인트 (베이스라인 대비)

| 항목 | 베이스라인 | 이 버전 | 효과 |
|------|-----------|--------|------|
| 캘리브레이션 샘플 | 256 | **512** | 양자화 품질 ↑ |
| 시퀀스 길이 | 512 | **1024** | 긴 문맥 학습 |
| actorder | (미지정) | **weight** | 정확도 ~2% ↑ |
| dampening_frac | (미지정) | **0.001** | Hessian 안정화 |

## 핵심 주의사항

**절대 변경 금지**:
```python
IGNORE = ["embed_tokens", "lm_head"]  # vLLM 호환 필수!
```

이 설정을 빼면 `tie_word_embeddings` 문제로 vLLM 에러 발생!

---